In [ ]:
# Export the labeled dataset
if labeled_df is not None:
    print("\\n" + "="*50)
    print("EXPORTING LABELED DATASET")
    print("="*50)
    
    # Create output directory
    output_dir = Path('./zebra_finch_labeled_features')
    output_dir.mkdir(exist_ok=True)
    
    # Save aggregated features (one row per file)
    agg_csv_path = output_dir / 'zebra_finch_labeled_aggregated.csv'
    labeled_df.to_csv(agg_csv_path, index=False)
    print(f"✓ Saved aggregated features: {agg_csv_path}")
    
    agg_pickle_path = output_dir / 'zebra_finch_labeled_aggregated.pkl'
    labeled_df.to_pickle(agg_pickle_path)
    print(f"✓ Saved aggregated pickle: {agg_pickle_path}")
    
    # Save raw features (one row per time frame)
    if raw_features_df is not None:
        raw_csv_path = output_dir / 'zebra_finch_labeled_raw_features.csv'
        raw_features_df.to_csv(raw_csv_path, index=False)
        print(f"✓ Saved raw features: {raw_csv_path}")
        
        raw_pickle_path = output_dir / 'zebra_finch_labeled_raw_features.pkl'
        raw_features_df.to_pickle(raw_pickle_path)
        print(f"✓ Saved raw features pickle: {raw_pickle_path}")
    
    # Save just the feature arrays for fast loading
    feature_cols = [col for col in labeled_df.columns if col.startswith('feature_')]
    
    # Aggregated features array
    agg_features_array = labeled_df[feature_cols].values
    agg_labels = labeled_df['call_type'].values
    
    np.save(output_dir / 'aggregated_features.npy', agg_features_array)
    np.save(output_dir / 'aggregated_labels.npy', agg_labels)
    print(f"✓ Saved aggregated arrays: features {agg_features_array.shape}, labels {agg_labels.shape}")
    
    # Raw features array (if available)
    if raw_features_df is not None:
        raw_features_array = raw_features_df[feature_cols].values
        raw_labels = raw_features_df['call_type'].values
        
        np.save(output_dir / 'raw_features.npy', raw_features_array)
        np.save(output_dir / 'raw_labels.npy', raw_labels)
        print(f"✓ Saved raw arrays: features {raw_features_array.shape}, labels {raw_labels.shape}")
    
    # Create label encoding mapping
    unique_labels = labeled_df['call_type'].unique()
    label_to_idx = {label: idx for idx, label in enumerate(sorted(unique_labels))}
    idx_to_label = {idx: label for label, idx in label_to_idx.items()}
    
    # Save label mappings
    import json
    with open(output_dir / 'label_mappings.json', 'w') as f:
        json.dump({
            'label_to_idx': label_to_idx,
            'idx_to_label': idx_to_label,
            'unique_labels': sorted(unique_labels)
        }, f, indent=2)
    print(f"✓ Saved label mappings: {len(unique_labels)} unique call types")
    
    # Create comprehensive report
    report_path = output_dir / 'labeled_dataset_report.txt'
    with open(report_path, 'w') as f:
        f.write("ZEBRA FINCH LABELED HUBERT DATASET REPORT\\n")
        f.write("=" * 50 + "\\n\\n")
        f.write(f"Processing Date: {pd.Timestamp.now()}\\n")
        f.write(f"Model: HuBERT Pretrain Base (768 dimensions)\\n")
        f.write(f"Audio Directory: {AUDIO_DIR}\\n")
        f.write(f"Checkpoint: {latest_checkpoint}\\n")
        f.write(f"Padding Length: {padding_length:.1f}s\\n\\n")
        
        f.write(f"Dataset Statistics:\\n")
        f.write(f"  Files processed successfully: {len(labeled_df)}\\n")
        f.write(f"  Files failed: {len(labeled_failures)}\\n")
        f.write(f"  Success rate: {len(labeled_df)/(len(labeled_df)+len(labeled_failures))*100:.1f}%\\n\\n")
        
        f.write(f"Call Type Distribution:\\n")
        call_type_counts = labeled_df['call_type'].value_counts()
        for call_type, count in call_type_counts.items():
            f.write(f"  {call_type}: {count}\\n")
        f.write(f"\\n")
        
        f.write(f"Duration Statistics:\\n")
        f.write(f"  Mean original duration: {labeled_df['original_duration'].mean():.2f}s\\n")
        f.write(f"  Files truncated: {labeled_df['was_truncated'].sum()}\\n")
        f.write(f"  Files padded: {(labeled_df['padding_added'] > 0).sum()}\\n\\n")
        
        f.write(f"Feature Statistics:\\n")
        f.write(f"  Dimensions: {len(feature_cols)}\\n")
        f.write(f"  Mean activation: {agg_features_array.mean():.4f}\\n")
        f.write(f"  Std activation: {agg_features_array.std():.4f}\\n")
        f.write(f"  Value range: {agg_features_array.min():.4f} to {agg_features_array.max():.4f}\\n\\n")
        
        if labeled_failures:
            f.write(f"Failed Files:\\n")
            for file_path, error in labeled_failures:
                f.write(f"  {file_path.name}: {error}\\n")
    
    print(f"✓ Saved comprehensive report: {report_path}")
    
    print(f"\\n📁 All labeled dataset files saved to: {output_dir}")
    print(f"\\n🎉 Labeled dataset creation complete!")
    print(f"   • {len(labeled_df)} audio files with call type labels")
    print(f"   • {len(call_type_counts)} different call types")
    print(f"   • {len(feature_cols)} HuBERT features per file")
    print(f"   • Ready for classification/clustering analysis!")
    
    # Quick access info
    print(f"\\n💡 Dataset summary:")
    print(f"   Aggregated features: {agg_features_array.shape} (one row per file)")
    if raw_features_df is not None:
        print(f"   Raw features: {raw_features_array.shape} (one row per time frame)")
    print(f"   Call types: {sorted(unique_labels)}")

else:
    print("❌ No labeled data to export - processing failed")

In [ ]:
def create_labeled_dataframe(results):
    """Create DataFrames for the labeled dataset.""" 
    if not results:
        print("No results to create DataFrame from")
        return None, None
    
    print(f"Creating labeled DataFrames from {len(results)} processed files...")
    
    # Get feature dimensions
    feature_dim = results[0]['feature_dim']
    print(f"Feature dimensionality: {feature_dim}")
    
    # Create aggregated features DataFrame (one row per file)
    aggregated_data = {
        'file_path': [],
        'filename': [],
        'call_type': [],
        'original_duration': [],
        'padded_duration': [],
        'was_truncated': [],
        'padding_added': [],
        'num_frames': [],
        'sample_rate': []
    }
    
    # Add feature columns
    feature_columns = [f'feature_{i}' for i in range(feature_dim)]
    for col in feature_columns:
        aggregated_data[col] = []
    
    # Fill aggregated data
    for result in results:
        aggregated_data['file_path'].append(result['file_path'])
        aggregated_data['filename'].append(result['filename'])
        aggregated_data['call_type'].append(result['call_type'])
        aggregated_data['original_duration'].append(result['original_duration'])
        aggregated_data['padded_duration'].append(result['padded_duration'])
        aggregated_data['was_truncated'].append(result['was_truncated'])
        aggregated_data['padding_added'].append(result['padding_added'])
        aggregated_data['num_frames'].append(result['num_frames'])
        aggregated_data['sample_rate'].append(result['sample_rate'])
        
        # Add aggregated feature values
        features = result['aggregated_features']
        for i, value in enumerate(features):
            aggregated_data[f'feature_{i}'].append(float(value))
    
    # Create aggregated DataFrame
    aggregated_df = pd.DataFrame(aggregated_data)
    
    # Create raw features array (time x features for each file)
    # This is more complex as files have different numbers of frames
    raw_features_list = []
    file_info_list = []
    
    for i, result in enumerate(results):
        raw_features = result['raw_features']  # [time_frames, feature_dim]
        num_frames = raw_features.shape[0]
        
        # Add file info for each frame
        for frame_idx in range(num_frames):
            file_info_list.append({
                'file_idx': i,
                'filename': result['filename'],
                'call_type': result['call_type'],
                'frame_idx': frame_idx,
                'time_sec': frame_idx * 0.02  # Assuming 50Hz frame rate (20ms per frame)
            })
            raw_features_list.append(raw_features[frame_idx])
    
    # Create raw features DataFrame
    raw_features_array = np.stack(raw_features_list)
    raw_data = pd.DataFrame(file_info_list)
    
    # Add feature columns to raw data
    for i in range(feature_dim):
        raw_data[f'feature_{i}'] = raw_features_array[:, i]
    
    print(f"\\nDataFrames created successfully!")
    print(f"Aggregated DataFrame shape: {aggregated_df.shape}")
    print(f"Raw features DataFrame shape: {raw_data.shape}")
    
    return aggregated_df, raw_data

# Create the labeled DataFrames
if labeled_results:
    labeled_df, raw_features_df = create_labeled_dataframe(labeled_results)
    
    if labeled_df is not None:
        print(f"\\n" + "="*50)
        print("LABELED DATASET SUMMARY")
        print("="*50)
        
        # Show basic info
        print(f"Total files processed: {len(labeled_df)}")
        call_type_counts = labeled_df['call_type'].value_counts()
        print(f"\\nCall type distribution:")
        for call_type, count in call_type_counts.items():
            print(f"  {call_type}: {count}")
        
        # Show duration statistics
        print(f"\\nDuration statistics:")
        print(f"  Mean original: {labeled_df['original_duration'].mean():.2f}s")
        print(f"  Files truncated: {labeled_df['was_truncated'].sum()}")
        print(f"  Files padded: {(labeled_df['padding_added'] > 0).sum()}")
        
        # Show feature statistics by call type
        feature_cols = [col for col in labeled_df.columns if col.startswith('feature_')]
        print(f"\\nFeature statistics by call type:")
        for call_type in call_type_counts.index[:3]:  # Show top 3 call types
            subset = labeled_df[labeled_df['call_type'] == call_type]
            feature_data = subset[feature_cols]
            print(f"  {call_type} (n={len(subset)}): mean={feature_data.mean().mean():.4f}, std={feature_data.std().mean():.4f}")
        
        # Show sample data
        print(f"\\nSample aggregated data:")
        sample_cols = ['filename', 'call_type', 'original_duration'] + feature_cols[:3]
        print(labeled_df[sample_cols].head(3).to_string(index=False))
        
else:
    print("No labeled results available to create DataFrame")
    labeled_df = None
    raw_features_df = None

In [ ]:
# Process full dataset if test successful
if test_results and len(audio_files) > 10:
    print("\\n" + "="*50)
    print("PROCESSING FULL LABELED DATASET")
    print("="*50)
    
    # Process all files
    labeled_results, labeled_failures = process_labeled_dataset(
        audio_files, 
        model, 
        padding_length,
        batch_size=8  # Adjust based on your GPU memory
    )
    
elif test_results:
    print("\\nUsing test results (only 10 files found)")
    labeled_results = test_results
    labeled_failures = test_failures
    
else:
    print("\\nCannot proceed - test processing failed")
    labeled_results = []
    labeled_failures = []

In [ ]:
def process_labeled_dataset(audio_files, model, padding_length, max_files=None, batch_size=8):
    """Process audio files with labels and consistent padding for batch processing."""
    if max_files:
        audio_files = audio_files[:max_files]
        print(f"Processing first {max_files} files for testing...")
    
    results = []
    failed_files = []
    
    print(f"Processing {len(audio_files)} files with {padding_length:.1f}s padding...")
    print(f"Batch size: {batch_size}")
    
    # Process in batches for efficiency
    for batch_start in tqdm(range(0, len(audio_files), batch_size), desc="Processing batches"):
        batch_files = audio_files[batch_start:batch_start + batch_size]
        batch_waveforms = []
        batch_metadata = []
        
        # Load and pad batch
        for file_path in batch_files:
            try:
                call_type = extract_call_label(file_path.name)
                waveform, sample_rate, original_duration = load_and_pad_audio(
                    file_path, 
                    target_length=padding_length
                )
                
                batch_waveforms.append(waveform)
                batch_metadata.append({
                    'file_path': str(file_path),
                    'filename': file_path.name,
                    'call_type': call_type,
                    'original_duration': original_duration,
                    'padded_duration': padding_length,
                    'sample_rate': sample_rate,
                    'was_truncated': original_duration > padding_length,
                    'padding_added': max(0, padding_length - original_duration)
                })
                
            except Exception as e:
                failed_files.append((file_path, str(e)))
                continue
        
        if not batch_waveforms:
            continue
            
        # Stack into batch tensor
        try:
            batch_tensor = torch.stack(batch_waveforms).to(device)  # [batch_size, 1, samples]
            batch_tensor = batch_tensor.squeeze(1)  # [batch_size, samples]
            
            # Extract features for the entire batch
            with torch.no_grad():
                if hasattr(model, 'extract_features'):
                    batch_features, _ = model.extract_features(batch_tensor)
                elif hasattr(model, 'wav2vec2'):
                    batch_features, _ = model.wav2vec2.extract_features(batch_tensor)
                else:
                    batch_features = model(batch_tensor)
                
                # Handle list of layer features
                if isinstance(batch_features, list):
                    batch_features = batch_features[-1]  # Use last layer
                
                # batch_features shape: [batch_size, time_frames, feature_dim]
                
            # Process each item in the batch
            for i, metadata in enumerate(batch_metadata):
                features = batch_features[i]  # [time_frames, feature_dim]
                
                # Store both raw features and aggregated features
                result = metadata.copy()
                result.update({
                    'success': True,
                    'raw_features': features.cpu().numpy(),  # Keep time dimension
                    'aggregated_features': features.mean(dim=0).cpu().numpy(),  # [feature_dim]
                    'num_frames': features.shape[0],
                    'feature_dim': features.shape[1]
                })
                results.append(result)
                
        except Exception as e:
            # If batch processing fails, mark all files in batch as failed
            for metadata in batch_metadata:
                failed_files.append((Path(metadata['file_path']), str(e)))
            continue
    
    print(f"\\nProcessing complete!")
    print(f"Successful: {len(results)}")
    print(f"Failed: {len(failed_files)}")
    
    if failed_files:
        print(f"\\nFirst few failed files:")
        for file_path, error in failed_files[:3]:
            print(f"  {file_path.name}: {error}")
    
    return results, failed_files

# Test with small batch first
print("\\n" + "="*50)
print("TESTING LABELED DATASET PROCESSING")
print("="*50)

test_results, test_failures = process_labeled_dataset(
    audio_files, 
    model, 
    padding_length, 
    max_files=10,  # Test with 10 files
    batch_size=4
)

if test_results:
    print(f"\\n✅ Test successful! Sample result:")
    sample = test_results[0]
    print(f"  File: {sample['filename']}")
    print(f"  Call type: {sample['call_type']}")
    print(f"  Original duration: {sample['original_duration']:.2f}s")
    print(f"  Padded duration: {sample['padded_duration']:.2f}s")
    print(f"  Was truncated: {sample['was_truncated']}")
    print(f"  Raw features shape: {sample['raw_features'].shape}")
    print(f"  Aggregated features shape: {sample['aggregated_features'].shape}")
    
    # Show call type distribution in test
    test_call_counts = Counter([r['call_type'] for r in test_results])
    print(f"\\n  Call types in test: {dict(test_call_counts)}")
else:
    print(f"\\n❌ Test failed!")
    if test_failures:
        print(f"First failure: {test_failures[0][1]}")

In [ ]:
def load_and_pad_audio(file_path, target_length=None, target_sample_rate=16000):
    """Load audio and pad/truncate to target length for batch processing."""
    # Load and preprocess audio
    waveform, sample_rate = torchaudio.load(file_path)
    
    # Resample if necessary
    if sample_rate != target_sample_rate:
        resampler = torchaudio.transforms.Resample(sample_rate, target_sample_rate)
        waveform = resampler(waveform)
    
    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    # Get the duration before padding
    original_duration = waveform.shape[1] / target_sample_rate
    
    # Pad or truncate to target length if specified
    if target_length is not None:
        target_samples = int(target_length * target_sample_rate)
        current_samples = waveform.shape[1]
        
        if current_samples < target_samples:
            # Pad with zeros
            padding = target_samples - current_samples
            waveform = torch.nn.functional.pad(waveform, (0, padding), mode='constant', value=0)
        elif current_samples > target_samples:
            # Truncate
            waveform = waveform[:, :target_samples]
    
    return waveform, target_sample_rate, original_duration

def determine_optimal_length(audio_files, sample_rate=16000, percentile=95):
    """Determine optimal padding length based on dataset statistics."""
    print(f"Analyzing audio lengths to determine optimal padding...")
    
    durations = []
    for i, file_path in enumerate(tqdm(audio_files[:100], desc="Sampling durations")):  # Sample first 100 files
        try:
            waveform, sr = torchaudio.load(file_path)
            duration = waveform.shape[1] / sr
            durations.append(duration)
        except:
            continue
    
    durations = np.array(durations)
    
    print(f"\\nDuration statistics (from {len(durations)} samples):")
    print(f"  Mean: {durations.mean():.2f}s")
    print(f"  Median: {np.median(durations):.2f}s")
    print(f"  Min: {durations.min():.2f}s")
    print(f"  Max: {durations.max():.2f}s")
    print(f"  {percentile}th percentile: {np.percentile(durations, percentile):.2f}s")
    
    # Use 95th percentile as padding length to cover most files
    optimal_length = np.percentile(durations, percentile)
    print(f"\\n📏 Recommended padding length: {optimal_length:.2f}s")
    print(f"   This will cover {percentile}% of files without truncation")
    
    return optimal_length

# Determine optimal padding length
optimal_length = determine_optimal_length(audio_files)

# Round up to nearest 0.5 seconds for cleaner processing
padding_length = np.ceil(optimal_length * 2) / 2
print(f"\\n🎯 Using padding length: {padding_length:.1f}s")

In [ ]:
import re
from collections import Counter

def extract_call_label(filename):
    """Extract call type from filename pattern: BirdID_Date-CallType-Number.wav"""
    # Pattern: anything-CALLTYPE-number.wav
    match = re.search(r'-([A-Za-z]+)-\d+[a-z]?\.wav$', filename)
    if match:
        return match.group(1)
    
    # Handle special cases like "Nest-C-13.wav" 
    match = re.search(r'-([A-Za-z]+-[A-Za-z]+)-\d+[a-z]?\.wav$', filename)
    if match:
        return match.group(1)
    
    # Handle Song files
    if '-Song-' in filename:
        return 'Song'
    
    return 'Unknown'

def analyze_call_types(audio_files):
    """Analyze the distribution of call types in the dataset."""
    call_counts = Counter()
    file_examples = {}
    
    for file_path in audio_files:
        call_type = extract_call_label(file_path.name)
        call_counts[call_type] += 1
        
        # Store first few examples for each call type
        if call_type not in file_examples:
            file_examples[call_type] = []
        if len(file_examples[call_type]) < 3:
            file_examples[call_type].append(file_path.name)
    
    print(f"Found {len(call_counts)} different call types:")
    print(f"{'Call Type':<15} {'Count':<8} {'Examples'}")
    print("-" * 70)
    
    for call_type, count in call_counts.most_common():
        examples = ", ".join(file_examples[call_type])
        print(f"{call_type:<15} {count:<8} {examples}")
    
    return call_counts, file_examples

# Analyze call types in the dataset
print("Analyzing call types in the dataset...")
call_counts, call_examples = analyze_call_types(audio_files)

# Remove 'Unknown' if it exists (files that don't match the pattern)
if 'Unknown' in call_counts:
    unknown_count = call_counts['Unknown']
    print(f"\\n⚠️  Warning: {unknown_count} files have unknown call types. Check filename patterns.")
else:
    print(f"\\n✓ All files have recognized call types!")

## 11. Labeled Dataset Creation with Padding

# HuBERT Checkpoint Evaluation

This notebook loads trained HuBERT checkpoints and provides tools for evaluating the pretraining pipeline.


## 1. Setup & Dependencies

In [34]:
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set paths - update these to match your actual checkpoint location
EXP_DIR = Path('./exp')
CHECKPOINT_DIR = EXP_DIR / 'checkpoints_librispeech_hubert_pretrain_base'

# If the above doesn't exist, try common alternatives
if not CHECKPOINT_DIR.exists():
    alternatives = [
        EXP_DIR / 'checkpoints_short_zebra_finch_hubert_pretrain_base',
        EXP_DIR / 'checkpoints_librilight_hubert_pretrain_base',
        Path('/Users/jonathanwang/Desktop/savioFolders/temp_train/checkpoints_short_zebra_finch_hubert_pretrain_base'),
    ]
    
    for alt in alternatives:
        if alt.exists():
            CHECKPOINT_DIR = alt
            break

print(f"Looking for checkpoints in: {CHECKPOINT_DIR}")
print(f"Checkpoint directory exists: {CHECKPOINT_DIR.exists()}")

# List all checkpoint directories in exp folder for debugging
if EXP_DIR.exists():
    checkpoint_dirs = [d for d in EXP_DIR.iterdir() if d.is_dir() and 'checkpoint' in d.name.lower()]
    if checkpoint_dirs:
        print(f"\nAvailable checkpoint directories:")
        for d in checkpoint_dirs:
            ckpt_files = list(d.glob('*.ckpt'))
            print(f"  {d.name} ({len(ckpt_files)} .ckpt files)")
    else:
        print(f"\nNo checkpoint directories found in {EXP_DIR}")
else:
    print(f"\nExp directory {EXP_DIR} does not exist")

Using device: cpu
Looking for checkpoints in: /Users/jonathanwang/Desktop/savioFolders/temp_train/checkpoints_short_zebra_finch_hubert_pretrain_base
Checkpoint directory exists: True

Exp directory exp does not exist


## 2. Model Loading Functions

In [35]:
def find_latest_checkpoint(checkpoint_dir):
    """Find the latest checkpoint file in the directory."""
    checkpoint_files = list(checkpoint_dir.glob('*.ckpt'))
    if not checkpoint_files:
        raise FileNotFoundError(f"No checkpoint files found in {checkpoint_dir}")
    
    # Sort by modification time and return the latest
    latest_checkpoint = max(checkpoint_files, key=lambda x: x.stat().st_mtime)
    print(f"Found {len(checkpoint_files)} checkpoint files")
    print(f"Latest checkpoint: {latest_checkpoint.name}")
    return latest_checkpoint

def load_hubert_from_lightning_checkpoint(checkpoint_path, model_name='hubert_pretrain_base', num_classes=100):
    """Load HuBERT model from Lightning checkpoint."""
    print(f"Loading checkpoint from: {checkpoint_path}")
    
    # Load the checkpoint
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    print("Checkpoint keys:", list(checkpoint.keys()))
    if 'state_dict' in checkpoint:
        print("Found state_dict in checkpoint")
        state_dict = checkpoint['state_dict']
        
        # Print some example keys to understand the structure
        print("\nExample state_dict keys:")
        for i, key in enumerate(list(state_dict.keys())[:10]):
            print(f"  {key}")
        if len(state_dict) > 10:
            print(f"  ... and {len(state_dict) - 10} more keys")
    
    # Create the model architecture
    if model_name == 'hubert_pretrain_base':
        model = torchaudio.models.hubert_pretrain_base(num_classes=num_classes)
    elif model_name == 'hubert_pretrain_large':
        model = torchaudio.models.hubert_pretrain_large(num_classes=num_classes)
    elif model_name == 'hubert_pretrain_xlarge':
        model = torchaudio.models.hubert_pretrain_xlarge(num_classes=num_classes)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    
    print(f"Created {model_name} model with {num_classes} classes")
    
    # Extract model weights from Lightning state dict
    model_state_dict = {}
    for key, value in state_dict.items():
        if key.startswith('model.'):
            new_key = key.replace('model.', '')
            model_state_dict[new_key] = value
    
    print(f"Extracted {len(model_state_dict)} model parameters")
    
    # Load the state dict
    try:
        model.load_state_dict(model_state_dict, strict=False)
        print("Successfully loaded model weights")
    except Exception as e:
        print(f"Error loading state dict: {e}")
        print("Attempting partial load...")
        model.load_state_dict(model_state_dict, strict=False)
    
    model = model.to(device)
    model.eval()
    
    return model, checkpoint

# Find and display available checkpoints
if CHECKPOINT_DIR.exists():
    try:
        latest_checkpoint = find_latest_checkpoint(CHECKPOINT_DIR)
        print(f"\nWill use checkpoint: {latest_checkpoint}")
    except FileNotFoundError as e:
        print(f"\nError: {e}")
        print("Please make sure you have trained checkpoints in the exp directory")
else:
    print(f"\nCheckpoint directory {CHECKPOINT_DIR} does not exist.")
    print("Please run training first or adjust the CHECKPOINT_DIR path.")

Found 1 checkpoint files
Latest checkpoint: epoch=0-step=71-v1.ckpt

Will use checkpoint: /Users/jonathanwang/Desktop/savioFolders/temp_train/checkpoints_short_zebra_finch_hubert_pretrain_base/epoch=0-step=71-v1.ckpt


## 3. Load the Model

In [36]:
# Load the model (adjust parameters as needed)
MODEL_NAME = 'hubert_pretrain_base'  # Change if you used a different model
NUM_CLASSES = 100  # Change to match your training configuration

try:
    model, checkpoint_info = load_hubert_from_lightning_checkpoint(
        latest_checkpoint, 
        model_name=MODEL_NAME, 
        num_classes=NUM_CLASSES
    )
    
    print("\nModel loaded successfully!")
    print(f"Model is on device: {next(model.parameters()).device}")
    
    # Print some checkpoint metadata if available
    if 'epoch' in checkpoint_info:
        print(f"Checkpoint epoch: {checkpoint_info['epoch']}")
    if 'global_step' in checkpoint_info:
        print(f"Global step: {checkpoint_info['global_step']}")
        
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nFalling back to pretrained model for demonstration...")
    
    # Fallback to pretrained model
    bundle = torchaudio.pipelines.HUBERT_BASE
    model = bundle.get_model().to(device)
    print("Loaded pretrained HuBERT model")

Loading checkpoint from: /Users/jonathanwang/Desktop/savioFolders/temp_train/checkpoints_short_zebra_finch_hubert_pretrain_base/epoch=0-step=71-v1.ckpt
Checkpoint keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers']
Found state_dict in checkpoint

Example state_dict keys:
  model.wav2vec2.feature_extractor.conv_layers.0.layer_norm.weight
  model.wav2vec2.feature_extractor.conv_layers.0.layer_norm.bias
  model.wav2vec2.feature_extractor.conv_layers.0.conv.weight
  model.wav2vec2.feature_extractor.conv_layers.1.conv.weight
  model.wav2vec2.feature_extractor.conv_layers.2.conv.weight
  model.wav2vec2.feature_extractor.conv_layers.3.conv.weight
  model.wav2vec2.feature_extractor.conv_layers.4.conv.weight
  model.wav2vec2.feature_extractor.conv_layers.5.conv.weight
  model.wav2vec2.feature_extractor.conv_layers.6.conv.weight
  model.wav2vec2.encoder.feature_projection.layer_norm.weight
  ... and 204 more keys
Cr

## 4. Basic Sanity Checks

In [37]:
def test_model_forward_pass(model, sample_rate=16000, duration=3.0):
    """Test that the model can process audio without errors."""
    print("Testing model forward pass...")
    
    # Create dummy audio data
    num_samples = int(sample_rate * duration)
    dummy_audio = torch.randn(1, num_samples).to(device)
    
    print(f"Input shape: {dummy_audio.shape}")
    print(f"Sample rate: {sample_rate} Hz")
    print(f"Duration: {duration} seconds")
    
    with torch.no_grad():
        try:
            if hasattr(model, 'extract_features'):
                # For HuBERT pretrain models - use extract_features method
                features, _ = model.extract_features(dummy_audio)
                # extract_features returns a list of features from different layers
                if isinstance(features, list):
                    print(f"✓ Feature extraction successful")
                    print(f"  Number of layers: {len(features)}")
                    print(f"  Last layer features shape: {features[-1].shape}")
                else:
                    print(f"✓ Feature extraction successful")
                    print(f"  Features shape: {features.shape}")
                return True
                
            elif 'HuBERTPretrainModel' in str(type(model)):
                # For HuBERT pretrain models that need labels
                print("Detected HuBERT pretraining model - testing feature extraction...")
                # Create dummy labels (sequence length will be determined by model)
                # First extract features to get the right sequence length
                if hasattr(model, 'wav2vec2'):
                    wav2vec_model = model.wav2vec2
                    features, _ = wav2vec_model.extract_features(dummy_audio)
                    
                    # Handle list of features from different layers
                    if isinstance(features, list):
                        print(f"✓ Feature extraction successful")
                        print(f"  Number of layers: {len(features)}")
                        print(f"  Last layer features shape: {features[-1].shape}")
                        last_layer_features = features[-1]
                    else:
                        print(f"✓ Feature extraction successful")
                        print(f"  Features shape: {features.shape}")
                        last_layer_features = features
                    
                    # Test with dummy labels for full forward pass
                    seq_len = last_layer_features.shape[1]
                    dummy_labels = torch.randint(0, 100, (1, seq_len)).to(device)
                    loss = model(dummy_audio, dummy_labels)
                    print(f"✓ Forward pass with labels successful")
                    print(f"  Loss: {loss.item():.4f}")
                    return True
                else:
                    print("  Could not find wav2vec2 submodule")
                    return False
                    
            else:
                # For regular HuBERT models
                output = model(dummy_audio)
                print(f"✓ Forward pass successful")
                if isinstance(output, tuple):
                    print(f"  Output shapes: {[x.shape for x in output]}")
                else:
                    print(f"  Output shape: {output.shape}")
                return True
            
        except Exception as e:
            print(f"✗ Forward pass failed: {e}")
            print(f"Model type: {type(model)}")
            print("Attempting feature extraction only...")
            
            # Try alternative feature extraction methods
            try:
                if hasattr(model, 'wav2vec2'):
                    features, _ = model.wav2vec2.extract_features(dummy_audio)
                    if isinstance(features, list):
                        print(f"✓ Feature extraction via wav2vec2 successful")
                        print(f"  Number of layers: {len(features)}")
                        print(f"  Last layer features shape: {features[-1].shape}")
                    else:
                        print(f"✓ Feature extraction via wav2vec2 successful")
                        print(f"  Features shape: {features.shape}")
                    return True
            except Exception as e2:
                print(f"✗ Alternative feature extraction failed: {e2}")
                return False

# Run the test
forward_pass_success = test_model_forward_pass(model)

if forward_pass_success:
    print("\n✓ Model is working correctly!")
else:
    print("\n✗ Model has issues - check the loading process")

Testing model forward pass...
Input shape: torch.Size([1, 48000])
Sample rate: 16000 Hz
Duration: 3.0 seconds
Detected HuBERT pretraining model - testing feature extraction...
✓ Feature extraction successful
  Number of layers: 12
  Last layer features shape: torch.Size([1, 149, 768])
✓ Forward pass with labels successful
✗ Forward pass failed: 'tuple' object has no attribute 'item'
Model type: <class 'torchaudio.models.wav2vec2.model.HuBERTPretrainModel'>
Attempting feature extraction only...
✓ Feature extraction via wav2vec2 successful
  Number of layers: 12
  Last layer features shape: torch.Size([1, 149, 768])

✓ Model is working correctly!


## 5. Audio Processing Pipeline

In [38]:
def load_and_preprocess_audio(file_path, target_sample_rate=16000):
    """Load audio file and preprocess it for HuBERT."""
    waveform, sample_rate = torchaudio.load(file_path)
    
    # Resample if necessary
    if sample_rate != target_sample_rate:
        resampler = torchaudio.transforms.Resample(sample_rate, target_sample_rate)
        waveform = resampler(waveform)
    
    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    
    return waveform, target_sample_rate

def extract_hubert_features(model, audio, sample_rate=16000):
    """Extract features from audio using the HuBERT model."""
    audio = audio.to(device)
    
    with torch.no_grad():
        if hasattr(model, 'extract_features'):
            # For HuBERT pretrain models with extract_features method
            features, _ = model.extract_features(audio)
            # extract_features returns a list of features from different layers
            if isinstance(features, list):
                return features[-1]  # Return the last (highest) layer features
            return features
        elif hasattr(model, 'wav2vec2'):
            # For HuBERT pretrain models with wav2vec2 submodule
            features, _ = model.wav2vec2.extract_features(audio)
            # Handle list of features from different layers
            if isinstance(features, list):
                return features[-1]  # Return the last (highest) layer features
            return features
        else:
            # For regular HuBERT models
            output = model(audio)
            if isinstance(output, tuple):
                return output[0]  # Usually the first element is the features
            return output

# Test with synthetic audio
print("Testing feature extraction with synthetic audio...")

# Create a synthetic audio signal (sine wave)
sample_rate = 16000
duration = 2.0
frequency = 440  # A4 note
t = torch.linspace(0, duration, int(sample_rate * duration))
synthetic_audio = torch.sin(2 * torch.pi * frequency * t).unsqueeze(0)

print(f"Synthetic audio shape: {synthetic_audio.shape}")
print(f"Duration: {duration} seconds")

# Extract features
features = extract_hubert_features(model, synthetic_audio, sample_rate)
print(f"Extracted features shape: {features.shape}")
print(f"Feature statistics:")
print(f"  Mean: {features.mean().item():.4f}")
print(f"  Std: {features.std().item():.4f}")
print(f"  Min: {features.min().item():.4f}")
print(f"  Max: {features.max().item():.4f}")

Testing feature extraction with synthetic audio...
Synthetic audio shape: torch.Size([1, 32000])
Duration: 2.0 seconds
Extracted features shape: torch.Size([1, 99, 768])
Feature statistics:
  Mean: -0.0000
  Std: 1.0000
  Min: -4.0833
  Max: 3.6659


## 6. Feature Visualization

In [ ]:
import re
from collections import Counter

def extract_call_label(filename):
    """Extract call type from filename pattern: BirdID_Date-CallType-Number.wav"""
    # Look for pattern: -CALLTYPE- where CALLTYPE is between two hyphens
    parts = filename.split('-')
    
    if len(parts) >= 3:
        # The call type should be the second-to-last part before the file number
        # e.g., "BlaBla0506_110302-DC-01.wav" -> "DC"
        # e.g., "BluRas07dd_110607-TukC-24.wav" -> "TukC"
        call_type = parts[-2]  # Second to last part (before the number)
        
        # Clean up any file extensions or numbers that might be attached
        call_type = re.sub(r'\d+[a-z]?$', '', call_type)  # Remove trailing numbers/letters
        call_type = re.sub(r'\.wav$', '', call_type)  # Remove .wav if somehow attached
        
        return call_type if call_type else 'Unknown'
    
    return 'Unknown'

def analyze_call_types(audio_files):
    """Analyze the distribution of call types in the dataset."""
    call_counts = Counter()
    file_examples = {}
    
    for file_path in audio_files:
        call_type = extract_call_label(file_path.name)
        call_counts[call_type] += 1
        
        # Store first few examples for each call type
        if call_type not in file_examples:
            file_examples[call_type] = []
        if len(file_examples[call_type]) < 3:
            file_examples[call_type].append(file_path.name)
    
    print(f"Found {len(call_counts)} different call types:")
    print(f"{'Call Type':<15} {'Count':<8} {'Examples'}")
    print("-" * 70)
    
    for call_type, count in call_counts.most_common():
        examples = ", ".join(file_examples[call_type])
        print(f"{call_type:<15} {count:<8} {examples}")
    
    return call_counts, file_examples

# Test the extraction function first
print("Testing call type extraction on sample filenames:")
test_files = [
    "BlaBla0506_110302-DC-01.wav",
    "BluRas07dd_110607-TukC-24.wav", 
    "GraGra0201_110620-AggC-02.wav",
    "BlaLbl8026_110421-Tet-03.wav",
    "BluRas61dd_110414-Song-08.wav",
    "BlaLbl8026_110609-WhineC-07.wav"
]

for filename in test_files:
    call_type = extract_call_label(filename)
    print(f"  {filename} -> {call_type}")

print(f"\n" + "="*50)

# Analyze call types in the actual dataset
print("Analyzing call types in the dataset...")
call_counts, call_examples = analyze_call_types(audio_files)

# Remove 'Unknown' if it exists (files that don't match the pattern)
if 'Unknown' in call_counts:
    unknown_count = call_counts['Unknown']
    print(f"\n⚠️  Warning: {unknown_count} files have unknown call types. Check filename patterns.")
    
    # Show some unknown examples for debugging
    unknown_files = [f for f in audio_files if extract_call_label(f.name) == 'Unknown']
    print("Unknown file examples:")
    for f in unknown_files[:5]:
        print(f"  {f.name}")
else:
    print(f"\n✓ All files have recognized call types!")

## 7. Evaluation Metrics

In [40]:
def analyze_model_representations(model, num_test_samples=5):
    """Analyze the quality of learned representations."""
    print("Analyzing model representations...")
    
    sample_rate = 16000
    duration = 3.0
    
    all_features = []
    
    # Generate diverse test signals
    test_signals = [
        # Sine waves at different frequencies
        torch.sin(2 * torch.pi * 220 * torch.linspace(0, duration, int(sample_rate * duration))),  # A3
        torch.sin(2 * torch.pi * 440 * torch.linspace(0, duration, int(sample_rate * duration))),  # A4
        torch.sin(2 * torch.pi * 880 * torch.linspace(0, duration, int(sample_rate * duration))),  # A5
        # White noise
        torch.randn(int(sample_rate * duration)) * 0.1,
        # Chirp signal (frequency sweep)
        torch.sin(2 * torch.pi * torch.linspace(100, 1000, int(sample_rate * duration)) * 
                 torch.linspace(0, duration, int(sample_rate * duration))),
    ]
    
    signal_names = ['220Hz Sine', '440Hz Sine', '880Hz Sine', 'White Noise', 'Chirp']
    
    for i, (signal, name) in enumerate(zip(test_signals[:num_test_samples], signal_names[:num_test_samples])):
        signal = signal.unsqueeze(0)  # Add batch dimension
        features = extract_hubert_features(model, signal, sample_rate)
        all_features.append(features.cpu())
        
        print(f"\n{name}:")
        print(f"  Features shape: {features.shape}")
        print(f"  Mean: {features.mean().item():.4f}")
        print(f"  Std: {features.std().item():.4f}")
        print(f"  Range: [{features.min().item():.4f}, {features.max().item():.4f}]")
    
    # Compute pairwise similarities
    print("\nPairwise feature similarities:")
    for i in range(len(all_features)):
        for j in range(i+1, len(all_features)):
            # Flatten features and compute cosine similarity
            feat1 = all_features[i].flatten()
            feat2 = all_features[j].flatten()
            
            cos_sim = torch.nn.functional.cosine_similarity(feat1, feat2, dim=0)
            print(f"  {signal_names[i]} vs {signal_names[j]}: {cos_sim.item():.4f}")
    
    return all_features

# Run the analysis
test_features = analyze_model_representations(model)

Analyzing model representations...

220Hz Sine:
  Features shape: torch.Size([1, 149, 768])
  Mean: -0.0000
  Std: 1.0000
  Range: [-4.1154, 3.6356]

440Hz Sine:
  Features shape: torch.Size([1, 149, 768])
  Mean: -0.0000
  Std: 1.0000
  Range: [-4.0779, 3.6554]

880Hz Sine:
  Features shape: torch.Size([1, 149, 768])
  Mean: 0.0000
  Std: 1.0000
  Range: [-3.4934, 3.8964]

White Noise:
  Features shape: torch.Size([1, 149, 768])
  Mean: -0.0000
  Std: 1.0000
  Range: [-4.3194, 4.0160]

Chirp:
  Features shape: torch.Size([1, 149, 768])
  Mean: 0.0000
  Std: 1.0000
  Range: [-4.1441, 3.9407]

Pairwise feature similarities:
  220Hz Sine vs 440Hz Sine: 0.6507
  220Hz Sine vs 880Hz Sine: 0.6175
  220Hz Sine vs White Noise: 0.6211
  220Hz Sine vs Chirp: 0.6406
  440Hz Sine vs 880Hz Sine: 0.5780
  440Hz Sine vs White Noise: 0.5999
  440Hz Sine vs Chirp: 0.6163
  880Hz Sine vs White Noise: 0.6096
  880Hz Sine vs Chirp: 0.6463
  White Noise vs Chirp: 0.6292


## 8. Interactive Exploration Tools

In [41]:
def process_custom_audio(file_path):
    """Process a custom audio file and visualize results."""
    try:
        print(f"Processing: {file_path}")
        
        # Load and preprocess
        waveform, sample_rate = load_and_preprocess_audio(file_path)
        print(f"Loaded audio: {waveform.shape}, sample rate: {sample_rate}")
        
        # Extract features
        features = extract_hubert_features(model, waveform, sample_rate)
        print(f"Extracted features: {features.shape}")
        
        # Visualize
        plt.figure(figsize=(15, 10))
        
        # Plot waveform
        plt.subplot(3, 1, 1)
        plt.plot(waveform.squeeze().numpy())
        plt.title(f'Waveform: {Path(file_path).name}')
        plt.xlabel('Samples')
        plt.ylabel('Amplitude')
        plt.grid(True, alpha=0.3)
        
        # Plot features
        plt.subplot(3, 1, 2)
        features_np = features.squeeze().cpu().numpy()
        plt.imshow(features_np.T, aspect='auto', origin='lower', cmap='viridis')
        plt.colorbar()
        plt.title('HuBERT Features')
        plt.xlabel('Time frames')
        plt.ylabel('Feature dimensions')
        
        # Plot feature statistics
        plt.subplot(3, 1, 3)
        plt.plot(features_np.mean(axis=1), label='Mean', alpha=0.8)
        plt.plot(features_np.std(axis=1), label='Std', alpha=0.8)
        plt.title('Feature Statistics Over Time')
        plt.xlabel('Time frames')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return features
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

def model_info():
    """Display detailed model information."""
    print("Model Information:")
    print(f"  Model type: {type(model).__name__}")
    print(f"  Device: {next(model.parameters()).device}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # Check if it's a pretrain model
    if hasattr(model, 'extract_features'):
        print("  Type: HuBERT Pretraining Model")
    else:
        print("  Type: HuBERT Fine-tuned Model")
    
    print("\nModel Architecture:")
    for name, module in model.named_children():
        print(f"  {name}: {type(module).__name__}")

# Display model information
model_info()

print("\n" + "="*50)
print("Interactive Tools Ready!")
print("="*50)
print("\nTo process a custom audio file, run:")
print("features = process_custom_audio('/path/to/your/audio.wav')")
print("\nTo get model info, run:")
print("model_info()")

Model Information:
  Model type: HuBERTPretrainModel
  Device: cpu
  Parameters: 94,594,176
  Trainable parameters: 94,594,176
  Type: HuBERT Fine-tuned Model

Model Architecture:
  wav2vec2: Wav2Vec2Model
  mask_generator: MaskGenerator
  logit_generator: LogitGenerator

Interactive Tools Ready!

To process a custom audio file, run:
features = process_custom_audio('/path/to/your/audio.wav')

To get model info, run:
model_info()


## 10. Batch Processing Zebra Finch Audio Files

In [43]:
import pandas as pd
import os
from tqdm import tqdm

# Set the audio directory path
AUDIO_DIR = Path('/Users/jonathanwang/Desktop/savioFolders/Zebra Finch Vocal Repertoires/AdultVocalizations')

def discover_audio_files(directory):
    """Scan directory for WAV files and return list of paths."""
    audio_extensions = ['.wav', '.WAV']
    audio_files = []

    if not directory.exists():
        print(f"Directory does not exist: {directory}")
        return []

    for ext in audio_extensions:
        audio_files.extend(list(directory.glob(f'**/*{ext}')))

    # Sort for consistent ordering
    audio_files.sort()

    print(f"Found {len(audio_files)} audio files in {directory}")
    if audio_files:
        print(f"First few files:")
        for i, file in enumerate(audio_files[:5]):
            print(f"  {i+1}. {file.name}")
        if len(audio_files) > 5:
            print(f"  ... and {len(audio_files) - 5} more files")

    return audio_files

# Discover all audio files
audio_files = discover_audio_files(AUDIO_DIR)

if len(audio_files) == 0:
    print("\\nNo audio files found. Please check the directory path.")
    print(f"Current path: {AUDIO_DIR}")
    print("\\nTrying to list contents of parent directory...")
    parent_dir = AUDIO_DIR.parent
    if parent_dir.exists():
        subdirs = [d for d in parent_dir.iterdir() if d.is_dir()]
        print(f"Subdirectories in {parent_dir}:")
        for subdir in subdirs[:10]:
            print(f"  {subdir.name}")
else:
    print(f"\\nReady to process {len(audio_files)} audio files.")

Found 2969 audio files in /Users/jonathanwang/Desktop/savioFolders/Zebra Finch Vocal Repertoires/AdultVocalizations
First few files:
  1. BlaBla0506_110302-AggC-04.wav
  2. BlaBla0506_110302-AggC-05.wav
  3. BlaBla0506_110302-DC-01.wav
  4. BlaBla0506_110302-DC-02.wav
  5. BlaBla0506_110302-DC-05.wav
  ... and 2964 more files
\nReady to process 2969 audio files.


In [44]:
def process_audio_file(file_path, model):
    """Process a single audio file and return features and metadata."""
    try:
        # Load and preprocess audio
        waveform, sample_rate = load_and_preprocess_audio(file_path)

        # Get duration in seconds
        duration = waveform.shape[1] / sample_rate

        # Extract features
        features = extract_hubert_features(model, waveform, sample_rate)

        # Aggregate features across time (mean pooling)
        if features.dim() == 3:  # [batch, time, features]
            aggregated_features = features.mean(dim=1).squeeze(0)  # [768]
            num_frames = features.shape[1]
        else:  # [time, features] or [features]
            if features.dim() == 2:
                aggregated_features = features.mean(dim=0)  # [768]
                num_frames = features.shape[0]
            else:
                aggregated_features = features
                num_frames = 1

        return {
            'success': True,
            'features': aggregated_features.cpu().numpy(),
            'duration': duration,
            'num_frames': num_frames,
            'sample_rate': sample_rate,
            'original_shape': waveform.shape
        }

    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")
        return {
            'success': False,
            'error': str(e),
            'features': None,
            'duration': None,
            'num_frames': None,
            'sample_rate': None,
            'original_shape': None
        }

def batch_process_audio_files(audio_files, model, max_files=None):
    """Process multiple audio files and return results."""
    if max_files:
        audio_files = audio_files[:max_files]
        print(f"Processing first {max_files} files for testing...")

    results = []
    failed_files = []

    print(f"Processing {len(audio_files)} audio files...")

    for i, file_path in enumerate(tqdm(audio_files, desc="Processing audio")):
        result = process_audio_file(file_path, model)

        if result['success']:
            # Add file information
            result['file_path'] = str(file_path)
            result['filename'] = file_path.name
            results.append(result)
        else:
            failed_files.append((file_path, result['error']))

        # Progress update every 50 files
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1}/{len(audio_files)} files. Success: {len(results)}, Failed: {len(failed_files)}")

    print(f"\\nProcessing complete!")
    print(f"Successful: {len(results)}")
    print(f"Failed: {len(failed_files)}")

    if failed_files:
        print(f"\\nFailed files:")
        for file_path, error in failed_files[:5]:  # Show first 5 failures
            print(f"  {file_path.name}: {error}")
        if len(failed_files) > 5:
            print(f"  ... and {len(failed_files) - 5} more failures")

    return results, failed_files

# Test with a small batch first
print("Testing with first 5 files...")
test_results, test_failures = batch_process_audio_files(audio_files, model, max_files=5)

if test_results:
    print(f"\\nTest successful! Sample result:")
    sample = test_results[0]
    print(f"  File: {sample['filename']}")
    print(f"  Duration: {sample['duration']:.2f}s")
    print(f"  Features shape: {sample['features'].shape}")
    print(f"  Feature range: [{sample['features'].min():.4f}, {sample['features'].max():.4f}]")
    print(f"  Feature mean: {sample['features'].mean():.4f}")
else:
    print("\\nTest failed - no files processed successfully")
    if test_failures:
        print("First failure:", test_failures[0][1])

Testing with first 5 files...
Processing first 5 files for testing...
Processing 5 audio files...


Processing audio: 100%|██████████| 5/5 [00:00<00:00, 20.47it/s]

\nProcessing complete!
Successful: 5
Failed: 0
\nTest successful! Sample result:
  File: BlaBla0506_110302-AggC-04.wav
  Duration: 0.31s
  Features shape: (768,)
  Feature range: [-2.3506, 2.6853]
  Feature mean: 0.0000


In [ ]:
# If test was successful, process all files
if test_results and len(audio_files) > 5:
    print("\\n" + "="*50)
    print("PROCESSING ALL AUDIO FILES")
    print("="*50)

    # Process all files
    all_results, all_failures = batch_process_audio_files(audio_files, model)

elif test_results:
    print("\\nUsing test results (only 5 files found)")
    all_results = test_results
    all_failures = test_failures

else:
    print("\\nCannot proceed - test processing failed")
    all_results = []
    all_failures = []

In [ ]:
def create_features_dataframe(results):
    """Create a pandas DataFrame from the processing results."""
    if not results:
        print("No results to create DataFrame from")
        return None

    print(f"Creating DataFrame from {len(results)} processed files...")

    # Extract feature dimensions (should be 768 for HuBERT base)
    feature_dim = len(results[0]['features'])
    print(f"Feature dimensionality: {feature_dim}")

    # Create column names for features
    feature_columns = [f'feature_{i}' for i in range(feature_dim)]

    # Create the main data structure
    data = {
        'file_path': [],
        'filename': [],
        'duration': [],
        'num_frames': [],
        'sample_rate': [],
    }

    # Add feature columns
    for col in feature_columns:
        data[col] = []

    # Fill the data
    for result in results:
        data['file_path'].append(result['file_path'])
        data['filename'].append(result['filename'])
        data['duration'].append(result['duration'])
        data['num_frames'].append(result['num_frames'])
        data['sample_rate'].append(result['sample_rate'])

        # Add feature values
        features = result['features']
        for i, value in enumerate(features):
            data[f'feature_{i}'].append(float(value))

    # Create DataFrame
    df = pd.DataFrame(data)

    print(f"\\nDataFrame created successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns[:10])}... (showing first 10)")

    return df

# Create the DataFrame
if all_results:
    features_df = create_features_dataframe(all_results)

    if features_df is not None:
        print(f"\\n" + "="*50)
        print("DATAFRAME SUMMARY")
        print("="*50)

        # Show basic info
        print(f"Total files processed: {len(features_df)}")
        print(f"Feature dimensions: {len([col for col in features_df.columns if col.startswith('feature_')])}")

        # Show duration statistics
        print(f"\\nAudio duration statistics:")
        print(f"  Mean: {features_df['duration'].mean():.2f}s")
        print(f"  Min: {features_df['duration'].min():.2f}s")
        print(f"  Max: {features_df['duration'].max():.2f}s")
        print(f"  Total: {features_df['duration'].sum():.2f}s ({features_df['duration'].sum()/60:.1f} minutes)")

        # Show feature statistics
        feature_cols = [col for col in features_df.columns if col.startswith('feature_')]
        feature_data = features_df[feature_cols]

        print(f"\\nFeature statistics:")
        print(f"  Mean of means: {feature_data.mean().mean():.4f}")
        print(f"  Std of means: {feature_data.mean().std():.4f}")
        print(f"  Global min: {feature_data.min().min():.4f}")
        print(f"  Global max: {feature_data.max().max():.4f}")

        # Show sample data
        print(f"\\nSample data (first 3 files, first 5 features):")
        sample_cols = ['filename', 'duration'] + feature_cols[:5]
        print(features_df[sample_cols].head(3).to_string(index=False))

else:
    print("No results available to create DataFrame")
    features_df = None

In [ ]:
# Export the data for future analysis
if features_df is not None:
    print("\\n" + "="*50)
    print("EXPORTING DATA")
    print("="*50)

    # Create output directory
    output_dir = Path('./zebra_finch_features')
    output_dir.mkdir(exist_ok=True)

    # Save as CSV
    csv_path = output_dir / 'zebra_finch_hubert_features.csv'
    features_df.to_csv(csv_path, index=False)
    print(f"✓ Saved CSV: {csv_path}")

    # Save as pickle (preserves data types and is faster to load)
    pickle_path = output_dir / 'zebra_finch_hubert_features.pkl'
    features_df.to_pickle(pickle_path)
    print(f"✓ Saved Pickle: {pickle_path}")

    # Save metadata only (smaller file for quick inspection)
    metadata_cols = ['file_path', 'filename', 'duration', 'num_frames', 'sample_rate']
    metadata_df = features_df[metadata_cols]
    metadata_path = output_dir / 'zebra_finch_metadata.csv'
    metadata_df.to_csv(metadata_path, index=False)
    print(f"✓ Saved Metadata: {metadata_path}")

    # Save just the feature vectors as numpy array (for fast loading in other tools)
    feature_cols = [col for col in features_df.columns if col.startswith('feature_')]
    feature_array = features_df[feature_cols].values
    numpy_path = output_dir / 'zebra_finch_features_768d.npy'
    np.save(numpy_path, feature_array)
    print(f"✓ Saved NumPy array: {numpy_path} (shape: {feature_array.shape})")

    # Create a summary report
    report_path = output_dir / 'processing_report.txt'
    with open(report_path, 'w') as f:
        f.write("ZEBRA FINCH HUBERT FEATURE EXTRACTION REPORT\\n")
        f.write("=" * 50 + "\\n\\n")
        f.write(f"Processing Date: {pd.Timestamp.now()}\\n")
        f.write(f"Model: HuBERT Pretrain Base (768 dimensions)\\n")
        f.write(f"Audio Directory: {AUDIO_DIR}\\n")
        f.write(f"Checkpoint: {latest_checkpoint}\\n\\n")

        f.write(f"Files processed successfully: {len(all_results)}\\n")
        f.write(f"Files failed: {len(all_failures)}\\n")
        f.write(f"Success rate: {len(all_results)/(len(all_results)+len(all_failures))*100:.1f}%\\n\\n")

        f.write(f"Audio Statistics:\\n")
        f.write(f"  Total duration: {features_df['duration'].sum():.1f} seconds ({features_df['duration'].sum()/60:.1f} minutes)\\n")
        f.write(f"  Mean duration: {features_df['duration'].mean():.2f} seconds\\n")
        f.write(f"  Duration range: {features_df['duration'].min():.2f} - {features_df['duration'].max():.2f} seconds\\n\\n")

        f.write(f"Feature Statistics:\\n")
        f.write(f"  Dimensions: {len(feature_cols)}\\n")
        f.write(f"  Mean activation: {feature_array.mean():.4f}\\n")
        f.write(f"  Std activation: {feature_array.std():.4f}\\n")
        f.write(f"  Value range: {feature_array.min():.4f} to {feature_array.max():.4f}\\n\\n")

        if all_failures:
            f.write(f"Failed Files:\\n")
            for file_path, error in all_failures:
                f.write(f"  {file_path.name}: {error}\\n")

    print(f"✓ Saved Processing Report: {report_path}")

    print(f"\\n📁 All files saved to: {output_dir}")
    print(f"\\n🎉 Feature extraction complete!")
    print(f"   • {len(features_df)} audio files processed")
    print(f"   • {len(feature_cols)} features per file")
    print(f"   • Ready for downstream analysis!")

    # Quick access variables for further analysis
    print(f"\\n💡 For further analysis, use:")
    print(f"   features_df: Complete DataFrame")
    print(f"   feature_array: Just the 768D feature vectors")
    print(f"   metadata_df: File information only")

else:
    print("❌ No data to export - feature extraction failed")